In [7]:
import cv2
import numpy as np

# Função auxiliar para segmentação HSV de tons de vermelho (Melhorada para Sombras)
def segmentar_vermelho(img_bgr):
    # Aplica um leve desfoque para homogeneizar a iluminação antes de ver a cor
    img_suavizada = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    hsv = cv2.cvtColor(img_suavizada, cv2.COLOR_BGR2HSV)
    
    # Expandimos os limites de S (Saturação) e V (Brilho/Valor) para pegar áreas escuras.
    # V desceu de 50 para 20, permitindo capturar o vermelho na sombra.
    lower1 = np.array([0, 30, 20])
    upper1 = np.array([10, 255, 255])
    
    # Expandimos o Hue (H) de 170 para 160 para pegar tons mais arroxeados/amarronzados da sombra
    lower2 = np.array([160, 30, 20])
    upper2 = np.array([180, 255, 255])
    
    mask1 = cv2.inRange(hsv, lower1, upper1)
    mask2 = cv2.inRange(hsv, lower2, upper2)
    
    return mask1 + mask2

def detectar_descontinuidade_borda(mascara_objeto, img_resultado):
    """
    Usa 'Defeitos de Convexidade' com filtro geométrico para ignorar sombras.
    """
    contornos, _ = cv2.findContours(mascara_objeto, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return False, np.zeros_like(mascara_objeto)

    maior_contorno = max(contornos, key=cv2.contourArea)
    
    # Suaviza o contorno para evitar pequenas distorções de pixel
    epsilon = 0.005 * cv2.arcLength(maior_contorno, True)
    contorno_suavizado = cv2.approxPolyDP(maior_contorno, epsilon, True)
    
    hull = cv2.convexHull(contorno_suavizado, returnPoints=False)
    
    descontinuidades = 0
    img_alertas = np.zeros_like(mascara_objeto)

    # Precisa de pontos suficientes para calcular os defeitos
    if hull is not None and len(hull) > 3 and len(contorno_suavizado) > 3:
        defeitos = cv2.convexityDefects(contorno_suavizado, hull)
        
        if defeitos is not None:
            for i in range(defeitos.shape[0]):
                s, e, f, d = defeitos[i, 0]
                profundidade = d / 256.0
                
                # Pegamos os pontos de início e fim do defeito
                start = tuple(contorno_suavizado[s][0])
                end = tuple(contorno_suavizado[e][0])
                ponto_fundo = tuple(contorno_suavizado[f][0])
                
                # Calcula a distância (largura) do buraco
                largura_defeito = np.sqrt((start[0] - end[0])**2 + (start[1] - end[1])**2)
                
                # CRITÉRIO DE DECISÃO:
                # Se for muito fundo (>15) E não for largo demais (<150 pixels), é uma quebra.
                # Se for muito largo, provavelmente é a máscara que falhou por causa da sombra.
                if profundidade > 15 and largura_defeito < 150:
                    descontinuidades += 1
                    cv2.circle(img_resultado, ponto_fundo, 15, (0, 0, 255), 3)
                    cv2.putText(img_resultado, "QUEBRA", (ponto_fundo[0] + 20, ponto_fundo[1]), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                    cv2.circle(img_alertas, ponto_fundo, 15, 255, -1)

    aprovado = (descontinuidades == 0)
    return aprovado, img_alertas
    
def analisar_roi_robusto(imagem_roi, sensibilidade_ajuste):
    img_resultado = imagem_roi.copy()
    
    # NOVO PASSO A: Isolar a peça por cor HSV (específica para o vaso vermelho)
    # Isso resolve a falha crítica de segmentação na foto.
    mascara_cor = segmentar_vermelho(imagem_roi)
    
    # Limpeza morfológica para criar uma máscara de objeto sólida
    kernel_limpeza = np.ones((7, 7), np.uint8)
    mascara_objeto = cv2.morphologyEx(mascara_cor, cv2.MORPH_OPEN, kernel_limpeza)
    mascara_objeto = cv2.morphologyEx(mascara_objeto, cv2.MORPH_CLOSE, kernel_limpeza)
    mascara_objeto = cv2.dilate(mascara_objeto, np.ones((5, 5), np.uint8), iterations=1) 
    
    # PASSO B: INSPEÇÃO DA BORDA EXTERNA (Mantido)
    aprovado_borda, img_alertas_borda = detectar_descontinuidade_borda(mascara_objeto, img_resultado)
    
    # PASSO C: INSPEÇÃO DA SUPERFÍCIE (Filtro Black-Hat)
    # O Black-Hat revela detalhes escuros (trincas) contra um fundo mais claro
    cinza = cv2.cvtColor(imagem_roi, cv2.COLOR_BGR2GRAY)
    kernel_blackhat = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    blackhat = cv2.morphologyEx(cinza, cv2.MORPH_BLACKHAT, kernel_blackhat)
    
    # Ajusta o limiar com base na sensibilidade
    # Sensibilidade mais alta -> limiar menor -> mais trincas detectadas
    limiar_base = 25 - sensibilidade_ajuste
    limiar_base = max(5, min(limiar_base, 255))
    
    _, trincas_binarias = cv2.threshold(blackhat, limiar_base, 255, cv2.THRESH_BINARY)
    
    # NOVO PASSO D: FILTRAGEM GEOMÉTRICA REVISADA
    # Limitamos a área de busca apenas ao objeto segmentado, sem erosão pesada.
    trincas_no_objeto = cv2.bitwise_and(trincas_binarias, mascara_objeto)
    
    contornos_trincas, _ = cv2.findContours(trincas_no_objeto, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mascara_trincas_finais = np.zeros_like(trincas_no_objeto)
    
    # Estatísticas para o painel de debug
    count_geometria_ok = 0
    
    for cnt in contornos_trincas:
        area_contorno = cv2.contourArea(cnt)
        if area_contorno > 10: # Ignora poeira
            rect = cv2.minAreaRect(cnt)
            (largura, altura) = rect[1]
            
            if largura > 0 and altura > 0:
                proporcao = max(largura, altura) / min(largura, altura)
                
                hull = cv2.convexHull(cnt)
                hull_area = cv2.contourArea(hull)
                solidity = area_contorno / hull_area if hull_area > 0 else 0
                
                # Critérios geométricos suavizados para validar trincas finas
                # Uma proporção de 1.5 e solidez menor que 0.9 valida trincas mais sutis.
                if proporcao > 1.5 and solidity < 0.9:
                    cv2.drawContours(mascara_trincas_finais, [cnt], -1, 255, thickness=cv2.FILLED)
                    count_geometria_ok += 1

    # PASSO E: CÁLCULO FINAL E DECISÃO
    area_falha_px = cv2.countNonZero(mascara_trincas_finais)
    
    # Tolerância estrita para trincas
    TOLERANCIA_MAX_PIXELS = 40      
    aprovado_superficie = (area_falha_px <= TOLERANCIA_MAX_PIXELS)
    
    aprovado_geral = aprovado_superficie and aprovado_borda

    # PASSO F: RETORNO E DEBUG
    img_resultado[mascara_trincas_finais == 255] = [0, 255, 255] # Destaca trincas em amarelo
    
    cv2.putText(img_resultado, f"Defeitos Superf.: {area_falha_px} px", (10, 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    
    imagens_debug = {
        "1. Mascara Cor (HSV)": mascara_cor,
        "2. Falhas Borda (Defeitos)": img_alertas_borda,
        "3. Trincas (Black-Hat)": blackhat,
        "4. Filtro Final Superfície (Revisado)": mascara_trincas_finais
    }
    
    return img_resultado, aprovado_geral, imagens_debug

def mostrar_painel_debug(imagens_debug):
    nomes = list(imagens_debug.keys())
    imgs_bgr = []
    
    for img in imagens_debug.values():
        if len(img.shape) == 2:
            imgs_bgr.append(cv2.cvtColor(img, cv2.COLOR_GRAY2BGR))
        else:
            imgs_bgr.append(img)
    
    for i in range(4):
        cv2.putText(imgs_bgr[i], nomes[i], (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
    linha1 = np.hstack((imgs_bgr[0], imgs_bgr[1]))
    linha2 = np.hstack((imgs_bgr[2], imgs_bgr[3]))
    painel = np.vstack((linha1, linha2))
    
    painel = cv2.resize(painel, (600, 600))
    cv2.imshow("Painel de Debug CV", painel)

def main():
    cap = cv2.VideoCapture(0)
    sensibilidade = 0
    analisar = False
    frame_analisado = None
    ultimo_debug = None
    
    print("--- SISTEMA DE INSPEÇÃO (SUPERFÍCIE + BORDAS) ---")
    print("Controles: 'Espaço' p/ Analisar | 'C' aumenta limiar | 'V' reduz limiar | 'ESC' p/ sair")
    
    largura_roi = 300
    altura_roi = 300
    
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        altura_tela, largura_tela = frame.shape[:2]
        x1 = (largura_tela - largura_roi) // 2
        y1 = (altura_tela - altura_roi) // 2
        x2 = x1 + largura_roi
        y2 = y1 + altura_roi
            
        if analisar and frame_analisado is not None:
            cv2.imshow("Inspecao de Qualidade", frame_analisado)
            if ultimo_debug:
                mostrar_painel_debug(ultimo_debug)
        else:
            frame_exibicao = frame.copy()
            cv2.rectangle(frame_exibicao, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame_exibicao, "Posicione a peca e aperte ESPACO", (x1 - 30, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
            cv2.imshow("Inspecao de Qualidade", frame_exibicao)
            
        tecla = cv2.waitKey(1) & 0xFF
        
        if tecla == 27: break
        elif tecla == 32: 
            if analisar: 
                analisar = False 
                cv2.destroyWindow("Painel de Debug CV") 
            else:
                roi = frame[y1:y2, x1:x2]
                roi_processado, aprovado, ultimo_debug = analisar_roi_robusto(roi, sensibilidade)
                
                frame_analisado = frame.copy()
                frame_analisado[y1:y2, x1:x2] = roi_processado
                
                cor = (0, 255, 0) if aprovado else (0, 0, 255)
                texto = "APROVADO" if aprovado else "REPROVADO"
                cv2.rectangle(frame_analisado, (x1, y1), (x2, y2), cor, 3)
                cv2.putText(frame_analisado, texto, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, cor, 3)
                
                analisar = True
                
        elif tecla == ord('c'): 
            sensibilidade += 1
            print(f"Sensibilidade (BlackHat Limiar): {sensibilidade}")
        elif tecla == ord('v'): 
            sensibilidade -= 1
            print(f"Sensibilidade (BlackHat Limiar): {sensibilidade}")

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

--- SISTEMA DE INSPEÇÃO (SUPERFÍCIE + BORDAS) ---
Controles: 'Espaço' p/ Analisar | 'C' aumenta limiar | 'V' reduz limiar | 'ESC' p/ sair


QFontDatabase: Cannot find font directory /home/ronaldo/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ronaldo/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ronaldo/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ronaldo/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Canno